# 🩺 Hi-CliTr: Cognitive Radiology Report Generation Demo

This notebook demonstrates how to use the trained model for generating radiology reports from chest X-ray images.

## 1. Setup

In [ ]:
# Install dependencies (run only once)
# !pip install -q torch torchvision transformers timm einops pillow

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
import numpy as np

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Load Model

In [ ]:
from models.model import create_model, CognitiveRadiologyModel
from config import get_config

# Initialize model
print('Loading model...')
model = create_model(pretrained=True, device=device)

# Load trained weights (if available)
checkpoint_path = '../checkpoints/best.pt'
try:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'Loaded checkpoint from epoch {checkpoint["epoch"]}')
except FileNotFoundError:
    print('No checkpoint found, using pretrained weights only')

model.eval()
print('Model ready!')

## 3. Image Preprocessing

In [ ]:
# Image transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_and_preprocess(image_path):
    """Load and preprocess a chest X-ray image"""
    image = Image.open(image_path).convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)
    return image, tensor

## 4. Generate Report from Sample Image

In [ ]:
# Load a sample image
# Replace with your own image path
sample_image_path = '../data/iu_xray/images/1_IM-0001-3001.png'

try:
    original_image, image_tensor = load_and_preprocess(sample_image_path)
    
    # Display image
    plt.figure(figsize=(8, 8))
    plt.imshow(original_image, cmap='gray')
    plt.title('Input Chest X-Ray')
    plt.axis('off')
    plt.show()
    
except FileNotFoundError:
    print(f'Image not found at {sample_image_path}')
    print('Creating a dummy tensor for demonstration...')
    image_tensor = torch.randn(1, 3, 224, 224).to(device)

In [ ]:
# Generate report
clinical_indication = "55-year-old male with fever and productive cough for 3 days"

with torch.no_grad():
    result = model.generate_report(
        images=image_tensor,
        indication=clinical_indication,
        max_findings_length=150,
        max_impression_length=75,
        return_labels=True
    )

# Display results
print('=' * 60)
print('GENERATED RADIOLOGY REPORT')
print('=' * 60)
print(f'\nClinical Indication: {clinical_indication}')
print('\n' + '-' * 60)
print(result['reports'][0])
print('-' * 60)

## 5. View Predicted CheXpert Labels

In [ ]:
# Get predicted labels with probabilities
labels = result['predicted_labels'][0].cpu().numpy()
label_names = result['label_names']

# Sort by probability
sorted_indices = np.argsort(labels)[::-1]

print('\nPredicted Pathologies (probability):')
print('-' * 40)
for idx in sorted_indices:
    prob = labels[idx]
    status = '✓' if prob > 0.5 else '✗'
    print(f'{status} {label_names[idx]:25s} {prob:.3f}')

In [ ]:
# Visualize as bar chart
plt.figure(figsize=(12, 6))
colors = ['green' if p > 0.5 else 'red' for p in labels]
plt.barh(range(len(labels)), labels, color=colors)
plt.yticks(range(len(labels)), label_names)
plt.xlabel('Probability')
plt.title('CheXpert Pathology Predictions')
plt.axvline(x=0.5, color='black', linestyle='--', label='Threshold')
plt.xlim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Batch Inference

In [ ]:
def generate_reports_batch(image_paths, indications):
    """Generate reports for multiple images"""
    results = []
    
    for path, indication in zip(image_paths, indications):
        try:
            _, tensor = load_and_preprocess(path)
        except:
            tensor = torch.randn(1, 3, 224, 224).to(device)
        
        with torch.no_grad():
            result = model.generate_report(
                images=tensor,
                indication=indication,
            )
        
        results.append({
            'image': path,
            'indication': indication,
            'report': result['reports'][0],
            'labels': result['predicted_labels'][0]
        })
    
    return results

# Example batch
# batch_results = generate_reports_batch(
#     ['image1.png', 'image2.png'],
#     ['Patient with chest pain', 'Follow-up for pneumonia']
# )

## 7. Model Architecture Summary

In [ ]:
# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Model Architecture Summary')
print('=' * 50)
print(f'PRO-FA Encoder:   {count_parameters(model.encoder):,} params')
print(f'MIX-MLP Classifier: {count_parameters(model.classifier):,} params')
print(f'RCTA Decoder:     {count_parameters(model.decoder):,} params')
print('-' * 50)
print(f'Total:            {count_parameters(model):,} params')
print(f'Trainable:        {count_trainable(model):,} params')

---

## Notes

- For best results, use high-quality chest X-ray images
- Provide detailed clinical indication for better context
- The model works best on PA (Posterior-Anterior) views
- Multi-view input (PA + Lateral) can improve accuracy

---

*Made with 🧠 by Team BrainDead*